In [1]:
import torch

residue_vocab = {'A': 0, 'U': 1, 'C': 2, 'G': 3}

def one_hot_encode_sequence(seq):
    # seq: list of characters like ['A', 'U', 'C']
    one_hot = torch.zeros(len(seq), 4)
    for i, res in enumerate(seq):
        if res in residue_vocab:
            one_hot[i, residue_vocab[res]] = 1
    return one_hot

In [2]:
from torch.utils.data import Dataset

class RNA3DDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe
        self.mol_ids = dataframe['ID'].apply(lambda x: "_".join(x.split('_')[:2])).unique()

    def __len__(self):
        return len(self.mol_ids)

    def __getitem__(self, idx):
        mol_id = self.mol_ids[idx]
        group = self.df[self.df['ID'].str.startswith(mol_id)].sort_values(by='resid')
        seq = group['resname'].tolist()
        coords = group[['x_1', 'y_1', 'z_1']].values
        x = one_hot_encode_sequence(seq)
        y = torch.tensor(coords, dtype=torch.float)
        return x, y


In [3]:
import torch.nn as nn

class RNA3DModel(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.coord_head = nn.Conv1d(hidden_dim, 3, kernel_size=1)  # Predict x,y,z

    def forward(self, x):
        # x: (batch, seq_len, 4)
        x = x.permute(0, 2, 1)  # -> (batch, 4, seq_len)
        features = self.encoder(x)
        coords = self.coord_head(features)  # -> (batch, 3, seq_len)
        coords = coords.permute(0, 2, 1)    # -> (batch, seq_len, 3)
        return coords


In [4]:
from torch.utils.data import DataLoader

dataset = RNA3DDataset(df)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

model = RNA3DModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(50):
    model.train()
    total_loss = 0
    for seqs, coords in loader:
        pred = model(seqs.float())
        loss = criterion(pred, coords)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


NameError: name 'df' is not defined

In [ ]:
model.eval()
with torch.no_grad():
    test_seq, _ = dataset[0]
    pred_coords = model(test_seq.unsqueeze(0).float())
    print(pred_coords)
